In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install japanize-matplotlib
!pip install optuna

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import gc
from itertools import product

# -------------------------------------------------------------------
# 1. データの読み込み (変更なし)
# -------------------------------------------------------------------
print("データの読み込みを開始します...")
try:
    train_df = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
    test_df = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
    stores_df = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
    oil_df = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv', parse_dates=['date'])
    holidays_df = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])
    print("データの読み込みが完了しました。")
except FileNotFoundError:
    print("エラー: データファイルが見つかりません。")
    exit()

# -------------------------------------------------------------------
# 2. 「売上0の日」を含む完全なデータセットを作成
# -------------------------------------------------------------------
print("「売上0の日」を含む完全なデータセットを作成します...")
train_dates = pd.date_range(train_df['date'].min(), '2017-08-15') # 訓練期間
test_dates = pd.date_range('2017-08-16', '2017-08-31') # テスト期間
all_dates = train_dates.union(test_dates)

all_stores = stores_df['store_nbr'].unique()
all_families = train_df['family'].unique()
all_combinations = product(all_dates, all_stores, all_families)

full_df = pd.DataFrame(all_combinations, columns=['date', 'store_nbr', 'family'])
full_df = pd.merge(full_df, train_df, on=['date', 'store_nbr', 'family'], how='left')
full_df = pd.merge(full_df, test_df, on=['date', 'store_nbr', 'family'], how='left')

full_df['sales'] = full_df['sales'].fillna(0)
# id_xとid_yを結合し、残ったNaNを0で埋める
full_df['id'] = full_df['id_y'].fillna(full_df['id_x']).fillna(0)
# その後で安全に整数型に変換
full_df['id'] = full_df['id'].astype(int)
full_df['onpromotion'] = full_df['onpromotion_y'].fillna(full_df['onpromotion_x']).fillna(0)
full_df.drop(['id_x', 'id_y', 'onpromotion_x', 'onpromotion_y'], axis=1, inplace=True)

del train_df, test_df; gc.collect()

# -------------------------------------------------------------------
# 3. 特徴量エンジニアリング
# -------------------------------------------------------------------
print("特徴量エンジニアリングを開始します...")

def create_base_features(df, stores_info, oil_prices, holidays_info):
    # 日付関連
    df['dayofweek'] = df['date'].dt.dayofweek
    df['dayofyear'] = df['date'].dt.dayofyear
    df['month'] = df['date'].dt.month
    
    # 【新テクニック①】日付特徴量の三角関数変換
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

    # 外部情報マージ
    oil_prices['dcoilwtico'] = oil_prices['dcoilwtico'].interpolate(method='linear', limit_direction='both')
    df = pd.merge(df, oil_prices, on='date', how='left')
    df = pd.merge(df, stores_info, on='store_nbr', how='left')
    holidays_info = holidays_info[holidays_info['transferred'] == False]
    df = pd.merge(df, holidays_info[['date', 'type']].rename(columns={'type':'holiday_type'}), on='date', how='left')

    # onpromotionの対数変換
    df['onpromotion'] = np.log1p(df['onpromotion'])
    
    # カテゴリカル変数のコード化
    for col in df.select_dtypes(include=['object', 'category']).columns:
        df[col] = df[col].astype('category').cat.codes
        
    return df

def create_ts_features(df):
    # 【新テクニック②】対数変換後の売上に対してラグ・移動平均を作成
    df['log1p_sales'] = np.log1p(df['sales'])
    
    lags = [7, 14, 28]
    windows = [7, 14]
    
    for lag in lags:
        df[f'log_sales_lag_{lag}'] = df.groupby(['store_nbr', 'family'])['log1p_sales'].transform(lambda x: x.shift(lag))
        
    for window in windows:
        df[f'log_sales_rolling_mean_{window}'] = df.groupby(['store_nbr', 'family'])['log1p_sales'].transform(lambda x: x.shift(7).rolling(window).mean())

    df = df.drop('log1p_sales', axis=1)
    return df

full_df = create_base_features(full_df, stores_df, oil_df, holidays_df)
full_df = create_ts_features(full_df)

del stores_df, oil_df, holidays_df; gc.collect()
print("特徴量エンジニアリングが完了しました。")

# -------------------------------------------------------------------
# 4. モデルの学習と予測
# -------------------------------------------------------------------
print("モデルの学習を開始します...")

# 訓練データとテストデータに分割
train_mask = full_df['date'] <= '2017-08-15'
test_mask = full_df['date'] > '2017-08-15'
train_processed = full_df[train_mask].copy()
test_processed = full_df[test_mask].copy()

# 検証期間を設定
val_start_date = '2017-08-01'
train_idx = train_processed['date'] < val_start_date
val_idx = train_processed['date'] >= val_start_date

features = [col for col in train_processed.columns if col not in ['date', 'id', 'sales']]
X_train = train_processed[train_idx][features]
y_train_log = np.log1p(train_processed[train_idx]['sales'])
X_val = train_processed[val_idx][features]
y_val_log = np.log1p(train_processed[val_idx]['sales'])

# パラメータ設定
lgb_params = {
    'objective': 'regression_l1', 'metric': 'rmse', 'n_estimators': 3000,
    'learning_rate': 0.05, 'feature_fraction': 0.7, 'bagging_fraction': 0.7,
    'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1,
    'num_leaves': 31, 'verbose': -1, 'n_jobs': -1, 'seed': 42,
    'boosting_type': 'gbdt',
}

model = lgb.LGBMRegressor(**lgb_params)
model.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(100, verbose=True)],
)

# スコア表示
best_score = model.best_score_['valid_0']['rmse']
print("\n" + "="*50)
print(f"✅ [新アプローチ] 最良の検証スコア (RMSLE): {best_score:.5f}")
print("="*50)

# --- 予測と提出ファイルの作成 ---
print("\n提出用ファイルの作成を開始します...")
predictions_log = model.predict(test_processed[features])
predictions = np.expm1(predictions_log)
predictions[predictions < 0] = 0

submission_df = pd.DataFrame({'id': test_processed['id'], 'sales': predictions})
submission_df.to_csv('submission.csv', index=False)
print("提出ファイル 'submission.csv' が正常に作成されました。🎉")